# Pegelonline Feeder (Fabric Notebook)

Polls the German WSV PegelOnline API for water-level measurements and emits CloudEvents to the bound Fabric Event Stream.

**Runtime model**
- The `pegelonline` package (with both generated producer sub-packages) is pre-installed by the **attached Fabric Environment**. No `%pip install` is required at runtime.
- The Event Stream connection string is **looked up at runtime** via the Fabric REST API using the notebook's workspace identity (or user identity). The deploy script does not bake any secret into the notebook.
- The bridge runs `pegelonline feed --once`, exits after one polling cycle, and persists dedupe state to a Lakehouse Files path so the next scheduled run only emits new measurements.

In [ ]:
# === PARAMETERS (overwritten by deploy-feeder-notebook.ps1) ===
EVENTSTREAM_NAME = "pegelonline-ingest"     # Fabric Event Stream item name in this workspace
STATE_FILE       = "/lakehouse/default/Files/feeder-state/pegelonline/state.json"
POLLING_INTERVAL = 900                       # seconds; only used if ONCE_MODE is False
ONCE_MODE        = True                      # True for scheduled execution
WORKSPACE_ID     = ""                        # filled in by deploy script (optional; falls back to runtime context)

## Look up the Event Stream connection string
Uses `notebookutils.credentials.getToken('pbi')` (workspace identity when scheduled, user identity when run interactively) to call the Fabric REST + ES MWC APIs and retrieve the custom-endpoint `primaryConnectionString`. The token is short-lived and never persisted.

In [ ]:
import os, time, json
import requests

FABRIC_API = 'https://api.fabric.microsoft.com/v1'

def _get_pbi_token():
    try:
        import notebookutils  # noqa: F401
        return notebookutils.credentials.getToken('pbi')
    except Exception:
        from notebookutils import mssparkutils
        return mssparkutils.credentials.getToken('pbi')

def _resolve_workspace_id(explicit: str) -> str:
    if explicit:
        return explicit
    try:
        import notebookutils
        ctx = notebookutils.runtime.context
        return ctx['currentWorkspaceId']
    except Exception:
        pass
    from notebookutils import mssparkutils
    ctx = json.loads(mssparkutils.runtime.context)
    return ctx['currentWorkspaceId']

def lookup_es_connection_string(workspace_id: str, eventstream_name: str) -> str:
    """Return the primary connection string for the named Event Stream's custom-endpoint source."""
    aad = _get_pbi_token()
    h_fabric = {'Authorization': f'Bearer {aad}', 'Content-Type': 'application/json'}

    ws = requests.get(f'{FABRIC_API}/workspaces/{workspace_id}', headers=h_fabric, timeout=30)
    ws.raise_for_status()
    capacity_id = ws.json().get('capacityId')
    if not capacity_id:
        raise RuntimeError(f'Workspace {workspace_id} has no capacityId (must be a Fabric capacity workspace).')

    es_list = requests.get(f'{FABRIC_API}/workspaces/{workspace_id}/eventstreams', headers=h_fabric, timeout=30)
    es_list.raise_for_status()
    es = next((x for x in es_list.json().get('value', []) if x['displayName'] == eventstream_name), None)
    if not es:
        raise RuntimeError(f"Event Stream '{eventstream_name}' not found in workspace {workspace_id}.")
    eventstream_id = es['id']

    cluster = requests.get(
        'https://api.powerbi.com/powerbi/globalservice/v201606/clusterDetails',
        headers={'Authorization': f'Bearer {aad}'}, timeout=15,
    )
    cluster.raise_for_status()
    cluster_host = cluster.json()['clusterUrl'].replace('https://', '').rstrip('/')

    mwc_body = {
        'workloadType': 'ES', 'workloadId': 'ES',
        'workspaceObjectId': workspace_id,
        'customerCapacityObjectId': capacity_id,
        'artifacts': [{'artifactObjectId': eventstream_id, 'artifactType': 'EventStream'}],
    }
    mwc = requests.post(
        f'https://{cluster_host}/metadata/v201606/generatemwctokenv2',
        headers={
            'Authorization': f'Bearer {aad}',
            'x-ms-orig-aad-token': aad,
            'x-ms-workload-resource-moniker': eventstream_id,
            'Content-Type': 'application/json',
        },
        json=mwc_body, timeout=30,
    )
    mwc.raise_for_status()
    mwc_token = mwc.json()['Token']
    target_host = mwc.json()['TargetUriHost']

    topo = requests.get(
        f'https://{target_host}/webapi/capacities/{capacity_id}/workloads/ES/ESService/Direct/v1/workspaces/{workspace_id}/artifacts/{eventstream_id}/topology',
        headers={'Authorization': f'MwcToken {mwc_token}'}, timeout=30,
    )
    topo.raise_for_status()
    sources = topo.json().get('sources', [])
    custom_ep = next((s for s in sources if s.get('type') == 'CustomEndpoint'), None)
    if not custom_ep:
        raise RuntimeError("Event Stream has no CustomEndpoint source.")
    datasource_id = custom_ep['id']

    last_err = None
    for attempt in range(5):
        try:
            r = requests.post(
                f'https://{target_host}/webapi/capacities/{capacity_id}/workloads/ES/ESService/Direct/v1/workspaces/{workspace_id}/artifacts/{eventstream_id}/datasource/{datasource_id}/keys',
                headers={'Authorization': f'MwcToken {mwc_token}', 'Content-Type': 'application/json'},
                data='{}', timeout=30,
            )
            r.raise_for_status()
            cs = r.json().get('primaryConnectionString')
            if cs:
                return cs
            last_err = 'empty primaryConnectionString'
        except Exception as e:
            last_err = str(e)
        time.sleep(min(15, 3 * (attempt + 1)))
    raise RuntimeError(f'Could not retrieve connection string after 5 attempts: {last_err}')

ws_id = _resolve_workspace_id(WORKSPACE_ID)
cs = lookup_es_connection_string(ws_id, EVENTSTREAM_NAME)
print(f"Workspace:        {ws_id}")
print(f"Event Stream:     {EVENTSTREAM_NAME}")
print(f"Connection ready: length={len(cs)}")
os.environ['CONNECTION_STRING'] = cs

## Prepare state and run one polling cycle

In [ ]:
import os, pathlib, sys

pathlib.Path(STATE_FILE).parent.mkdir(parents=True, exist_ok=True)
os.environ['STATE_FILE']       = STATE_FILE
os.environ['POLLING_INTERVAL'] = str(POLLING_INTERVAL)
os.environ['ONCE_MODE']        = 'true' if ONCE_MODE else 'false'

# pegelonline + its 2 producer sub-packages come from the attached Fabric Environment.
from pegelonline import pegelonline as feeder

argv_backup = sys.argv
try:
    sys.argv = ['pegelonline', 'feed', '--once'] if ONCE_MODE else ['pegelonline', 'feed']
    feeder.main()
finally:
    sys.argv = argv_backup
print('Cycle complete.')